## Day 9 — Final Pipeline and Model Persistence

In [2]:
import os

# Bulunduğun konumu bir üst klasöre taşır
os.chdir("..")

# Şu an hangi klasörde olduğunu doğrulamak için:
print(os.getcwd())

/Users/nihalapple/Desktop/trend-forecast-project


In [3]:
from src.fetch_data import fetch_trends_data

In [5]:
test_data = fetch_trends_data(
    keywords=["chatgpt"],
    geo="",
    timeframe="today 3-y",
)

test_data.tail()

RuntimeError: Google Trends isteği başarısız oldu. İstek parametrelerini veya pytrends uyumluluğunu kontrol edin.

In [6]:
import importlib
import src.fetch_data

importlib.reload(src.fetch_data)

<module 'src.fetch_data' from '/Users/nihalapple/Desktop/trend-forecast-project/src/fetch_data.py'>

In [7]:
from src.fetch_data import fetch_trends_data

In [8]:
test_data = fetch_trends_data(
    keywords=["chatgpt"],
    geo="",
    timeframe="today 3-y",
)

test_data.tail()

ResponseError type: ResponseError
ResponseError detail: The request failed: Google returned a response with code 400


RuntimeError: Google Trends isteği başarısız oldu. İstek parametrelerini veya pytrends uyumluluğunu kontrol edin.

## Day 9 — Updating Weekly Google Trends Data

Pytrends üzerinden veri güncelleme denemesinde Google Trends tarafından
`400 ResponseError` döndürüldüğü için güncel veriler Google Trends web
arayüzünden manuel olarak indirildi.

Google Trends web arayüzünde uzun tarih aralıkları aylık frekansta
indirilebildiği için son 1 yıllık dönem ayrıca seçildi ve haftalık veri elde
edildi.

Ancak Google Trends web arayüzü her sorguda sonuçları yeniden 0–100
aralığında ölçeklendirdiğinden, farklı zaman aralıklarında indirilen CSV
dosyalarının değerleri doğrudan birleştirilemez.

Örneğin aynı tarih için eski 3 yıllık veri setindeki ChatGPT değeri ile yeni
1 yıllık veri setindeki ChatGPT değeri farklı olabilir. Bu fark gerçek arama
ilgisinin değişmesinden değil, sorguların farklı ölçeklerde normalize
edilmesinden kaynaklanabilir.

Bu nedenle mevcut 3 yıllık haftalık veri seti ana referans olarak korunacaktır.
Yeni 1 yıllık veri seti ile eski veri setinin ortak haftaları kullanılarak
ölçek farkı hesaplanacaktır.

İzlenecek yöntem:

1. Eski ve yeni haftalık veri setleri okunacak.
2. Her iki veri setinde bulunan ortak tarihler belirlenecek.
3. Ortak haftalardaki ChatGPT, Gemini ve Claude değerleri karşılaştırılacak.
4. Yeni veri setini eski veri setinin ölçeğine taşımak için ölçek faktörü
   hesaplanacak.
5. Ölçeklendirme doğrulandıktan sonra yalnızca eski veri setinde bulunmayan
   yeni haftalar eklenecek.
6. Güncellenmiş haftalık veri yeni bir processed CSV olarak kaydedilecek.

Bu yöntemle mevcut geçmiş veri korunurken yeni haftaların aynı ölçek üzerinde
seriye eklenmesi amaçlanmaktadır.

Buradaki temel problem gerçekten Google Trends’in web sitesinde her veri isteğini yeniden 0–100 ölçeğine getirmesi. Google da bu nedenle web sitesinden farklı isteklerde alınan parçaların doğrudan birleştirilemediğini, yeni Trends API’nin ise “consistently scaled” veri sunacağını belirtiyor.

In [9]:
# --------------------------------------------------
# Eski 3 yıllık haftalık veri
# --------------------------------------------------

import pandas as pd


old_data = pd.read_csv(
    "../data/processed/google_trends_ai_3y_clean.csv",
    parse_dates=["date"],
    index_col="date",
)


# --------------------------------------------------
# Yeni 1 yıllık manuel Google Trends verisi
# --------------------------------------------------

new_data = pd.read_csv(
    "../data/raw/google_trends_ai_1y_manual_2026-08-13.csv"
)


print("Old data:")
display(old_data.head())

print("\nNew data:")
display(new_data.head())

print("\nNew columns:")
print(new_data.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/google_trends_ai_3y_clean.csv'

In [10]:
import os

print(os.getcwd())

/Users/nihalapple/Desktop/trend-forecast-project


In [11]:
print(
    os.path.exists(
        "../data/processed/google_trends_ai_3y_clean.csv"
    )
)

False


### 1. Eski ve yeni veriyi okuyalım

In [12]:
old_data = pd.read_csv(
    "data/processed/google_trends_ai_3y_clean.csv",
    parse_dates=["date"],
    index_col="date",
)

In [13]:
new_data = pd.read_csv(
    "data/raw/google_trends_ai_1y_manual_2026-08-13.csv"
)

In [14]:
print("Old data:")
display(old_data.head())

print("\nNew data:")
display(new_data.head())

print("\nNew columns:")
print(new_data.columns.tolist())

Old data:


,chatgpt,gemini,claude
date,,,
2023-07-30,13,1,1
2023-08-06,14,1,1
2023-08-13,14,1,1
2023-08-20,15,1,1
2023-08-27,16,1,1



New data:


,Time,chatgpt,gemini,claude
0,2025-08-10,73,12,4
1,2025-08-17,78,12,4
2,2025-08-24,80,13,4
3,2025-08-31,79,28,3
4,2025-09-07,82,63,3



New columns:
['Time', 'chatgpt', 'gemini', 'claude']


### 2. Yeni dosyanın sütunlarını bizim formata çevirelim

In [15]:
new_data = new_data.rename(
    columns={
        "Time": "date",
    }
)

new_data["date"] = pd.to_datetime(
    new_data["date"]
)

new_data = new_data.set_index("date")

In [18]:
print("\nNew columns:")
print(old_data.columns.tolist())


New columns:
['chatgpt', 'gemini', 'claude']


In [17]:
print("\nNew columns:")
print(new_data.columns.tolist())


New columns:
['chatgpt', 'gemini', 'claude']


In [16]:
print("Old range:")
print(old_data.index.min(), "→", old_data.index.max())

print("\nNew range:")
print(new_data.index.min(), "→", new_data.index.max())

print("\nOld shape:", old_data.shape)
print("New shape:", new_data.shape)

Old range:
2023-07-30 00:00:00 → 2026-07-26 00:00:00

New range:
2025-08-10 00:00:00 → 2026-08-09 00:00:00

Old shape: (157, 3)
New shape: (53, 3)


### 3. Ortak haftaları bulalım

In [ ]:
common_dates = old_data.index.intersection(
    new_data.index
)

print(type(old_data.index))

print("Ortak hafta sayısı:", len(common_dates))
print("İlk ortak tarih:", common_dates.min())
print("Son ortak tarih:", common_dates.max())

<class 'pandas.DatetimeIndex'>
Ortak hafta sayısı: 51
İlk ortak tarih: 2025-08-10 00:00:00
Son ortak tarih: 2026-07-26 00:00:00


ortak kısımları yan yana getirelim

In [21]:
overlap = pd.DataFrame(
    {
        "old_chatgpt": old_data.loc[
            common_dates, "chatgpt"
        ],

        "new_chatgpt": new_data.loc[
            common_dates, "chatgpt"
        ],

        "old_gemini": old_data.loc[
            common_dates, "gemini"
        ],

        "new_gemini": new_data.loc[
            common_dates, "gemini"
        ],

        "old_claude": old_data.loc[
            common_dates, "claude"
        ],

        "new_claude": new_data.loc[
            common_dates, "claude"
        ],
    }
)


overlap.head(10)

,old_chatgpt,new_chatgpt,old_gemini,new_gemini,old_claude,new_claude
date,,,,,,
2025-08-10,72,73,12,12,4,4
2025-08-17,75,78,12,12,4,4
2025-08-24,79,80,13,13,4,4
2025-08-31,79,79,28,28,3,3
2025-09-07,85,82,63,63,3,3
2025-09-14,91,90,100,100,3,3
2025-09-21,79,83,60,59,3,3
2025-09-28,82,77,45,44,4,4
2025-10-05,85,83,37,37,4,3


### 4. Ölçek oranlarını hesaplayalım

In [22]:
#Şimdilik her seri için ayrı ayrı oranların medyanına bakalım:

scale_factors = {}


for keyword in [
    "chatgpt",
    "gemini",
    "claude",
]:

    old_values = overlap[
        f"old_{keyword}"
    ]

    new_values = overlap[
        f"new_{keyword}"
    ]


    # 0'a bölme problemi oluşmasın
    valid = new_values > 0


    ratios = (
        old_values[valid]
        / new_values[valid]
    )


    scale_factors[keyword] = ratios.median()


scale_factors#

{'chatgpt': np.float64(1.0161290322580645),
 'gemini': np.float64(1.0),
 'claude': np.float64(1.0)}

In [23]:
# --------------------------------------------------
# Ortak haftalardaki ölçek oranlarını incele
# --------------------------------------------------

for keyword in [
    "chatgpt",
    "gemini",
    "claude",
]:
    old_values = overlap[f"old_{keyword}"]
    new_values = overlap[f"new_{keyword}"]

    # 0'a bölmeyi önle
    valid = new_values > 0

    ratios = (
        old_values[valid]
        / new_values[valid]
    )

    print(f"\n{keyword.upper()}")
    print(ratios.describe())


CHATGPT
count    51.000000
mean      1.020025
std       0.027592
min       0.951807
25%       1.000000
50%       1.016129
75%       1.038841
max       1.078125
dtype: float64

GEMINI
count    51.000000
mean      1.012988
std       0.025596
min       0.944444
25%       1.000000
50%       1.000000
75%       1.028665
max       1.096774
dtype: float64

CLAUDE
count    51.000000
mean      1.023227
std       0.069996
min       0.944444
25%       1.000000
50%       1.000000
75%       1.000000
max       1.333333
dtype: float64


Burada özellikle:

mean
std
min
50%   ← median
max

değerlerine bakacağız

In [24]:
# Bir de daha anlaşılır olması için eski ve yeni değerlerin farkını görelim:

for keyword in [
    "chatgpt",
    "gemini",
    "claude",
]:
    overlap[f"{keyword}_difference"] = (
        overlap[f"old_{keyword}"]
        - overlap[f"new_{keyword}"]
    )

overlap[
    [
        "old_chatgpt",
        "new_chatgpt",
        "chatgpt_difference",
        "old_gemini",
        "new_gemini",
        "gemini_difference",
        "old_claude",
        "new_claude",
        "claude_difference",
    ]
].tail(15)

,old_chatgpt,new_chatgpt,chatgpt_difference,old_gemini,new_gemini,gemini_difference,old_claude,new_claude,claude_difference
date,,,,,,,,,
2026-04-19,73,71,2,41,41,0,19,18,1
2026-04-26,71,69,2,43,42,1,17,17,0
2026-05-03,73,72,1,44,44,0,17,18,-1
2026-05-10,72,70,2,44,45,-1,18,18,0
2026-05-17,73,74,-1,46,47,-1,18,18,0
2026-05-24,70,69,1,44,43,1,19,19,0
2026-05-31,74,71,3,45,43,2,19,19,0
2026-06-07,71,72,-1,47,47,0,21,21,0
2026-06-14,70,68,2,43,43,0,19,19,0


Bu sonuçlar aslında iki dosyanın ölçeğinin çok yakın olduğunu gösteriyor. Özellikle ChatGPT ve Gemini’de dağılım oldukça dar:

ChatGPT
median = 1.016
std    = 0.028

Gemini
median = 1.000
std    = 0.026

Yani ortak 51 haftanın büyük bölümünde yeni ve eski veri birbirine çok yakın.

Claude’daki max = 1.333 ilk bakışta korkutmasın. Claude değerleri zaten daha küçük sayılar olduğu için örneğin eski dosyada 4, yeni dosyada 3 olması bile:

4 / 3 = 1.333

oranını oluşturuyor. Yani burada büyük bir ölçek bozukluğundan ziyade küçük tam sayıların oran hesabını büyütmesi söz konusu.

Ama burada bir noktayı iyileştirelim: Google Trends’te ChatGPT, Gemini ve Claude aynı sorguda karşılaştırıldığı için üçüne tamamen ayrı scale factor uygulamak yerine ortak overlap bölgesinden tek sağlam bir ölçek faktörü çıkarmamız daha mantıklı.

In [25]:
import numpy as np


#Şimdi bunu hesaplayalım:

# --------------------------------------------------
# Tüm keyword'leri kullanarak ortak scale factor
# --------------------------------------------------

all_ratios = []

for keyword in [
    "chatgpt",
    "gemini",
    "claude",
]:
    old_values = overlap[f"old_{keyword}"]
    new_values = overlap[f"new_{keyword}"]

    valid = (
        (old_values > 0)
        & (new_values > 0)
    )

    ratios = (
        old_values[valid]
        / new_values[valid]
    )

    all_ratios.extend(
        ratios.tolist()
    )


global_scale_factor = np.median(
    all_ratios
)

print(
    "Global scale factor:",
    global_scale_factor,
)

Global scale factor: 1.0


Güzel. global_scale_factor = 1.0 çıkması şu anlama geliyor:

Ortak haftaların tamamına baktığımızda yeni dosyanın eski dosyaya göre sistematik olarak daha yüksek ya da daha düşük bir ölçeği yok.

Yani genel olarak:

eski ölçek ≈ yeni ölçek

Bu yüzden yeni veriyi × 0.84, × 1.1 gibi bir katsayıyla düzeltmemize gerek görünmüyor.

Ama senin gördüğün 87 → 73 gibi tekil farklar hâlâ olabilir. scale factor = 1.0, her haftanın aynı olduğu anlamına gelmez; sadece farkların genel olarak tek yönde olmadığını gösterir. O yüzden eklemeden önce son bir kontrol yapalım.

In [26]:
# --------------------------------------------------
# Ortak haftalardaki farkları kontrol et
# --------------------------------------------------

for keyword in [
    "chatgpt",
    "gemini",
    "claude",
]:
    difference = (
        overlap[f"old_{keyword}"]
        - overlap[f"new_{keyword}"]
    ).abs()

    print(f"\n{keyword.upper()}")
    print("Mean absolute difference:", difference.mean())
    print("Median absolute difference:", difference.median())
    print("Maximum difference:", difference.max())


CHATGPT
Mean absolute difference: 1.9215686274509804
Median absolute difference: 2.0
Maximum difference: 5

GEMINI
Mean absolute difference: 0.7058823529411765
Median absolute difference: 1.0
Maximum difference: 3

CLAUDE
Mean absolute difference: 0.21568627450980393
Median absolute difference: 0.0
Maximum difference: 1


Bir de en fazla farklılaşan haftaları görelim:

In [27]:
# --------------------------------------------------
# En büyük farkların olduğu haftaları incele
# --------------------------------------------------

for keyword in [
    "chatgpt",
    "gemini",
    "claude",
]:
    comparison = pd.DataFrame(
        {
            "old": overlap[f"old_{keyword}"],
            "new": overlap[f"new_{keyword}"],
        }
    )

    comparison["difference"] = (
        comparison["old"]
        - comparison["new"]
    ).abs()

    print(f"\n{keyword.upper()} - Largest Differences")

    display(
        comparison.sort_values(
            by="difference",
            ascending=False,
        ).head(5)
    )


CHATGPT - Largest Differences


,old,new,difference
date,,,
2026-03-29,69,64,5
2025-09-28,82,77,5
2025-12-14,75,70,5
2026-07-19,69,65,4
2026-06-28,65,61,4



GEMINI - Largest Differences


,old,new,difference
date,,,
2026-02-01,34,31,3
2026-03-22,42,40,2
2026-03-15,39,37,2
2026-03-08,41,39,2
2026-02-22,36,34,2



CLAUDE - Largest Differences


,old,new,difference
date,,,
2026-07-26,16,15,1
2026-04-19,19,18,1
2026-07-19,16,15,1
2026-06-28,18,17,1
2026-03-15,15,14,1


Bu sonuçlara bakınca artık daha net karar verebiliriz. İki veri seti aynı ölçeğe çok yakın, ama birebir aynı değil:

ChatGPT → ortalama fark ≈ 1.92, maksimum 5
Gemini  → ortalama fark ≈ 0.71, maksimum 3
Claude  → ortalama fark ≈ 0.22, maksimum 1

Yani burada büyük bir ölçek kırılması yok. Ayrıca senin başta gördüğün 87 vs 73 gibi 14 puanlık fark bu kullandığımız doğru 1 yıllık overlap dosyasında görünmüyor; o büyük fark başka tarih aralığıyla oluşturulan sorgudan kaynaklanmış olmalı.

Ama bir şeyi iyileştirelim: global_scale_factor = 1.0 sonucunu direkt kullanıp geçmeyelim. Çünkü median hesaplamasında Claude gibi küçük ve sık sık birebir aynı olan değerler 1.0 sonucunu fazla etkileyebilir. Google Trends'te üç kelimeyi aynı sorguda çektiğimiz için teorik olarak hepsine tek ortak çarpan uygulamak daha mantıklı.

Bunu overlap verisinden daha sağlam şekilde hesaplayalım:

In [28]:
# --------------------------------------------------
# Ortak ölçek faktörünü daha sağlam hesapla
# old ≈ scale_factor * new
# --------------------------------------------------

keywords = [
    "chatgpt",
    "gemini",
    "claude",
]

old_values = np.concatenate(
    [
        overlap[f"old_{keyword}"].to_numpy(
            dtype=float
        )
        for keyword in keywords
    ]
)

new_values = np.concatenate(
    [
        overlap[f"new_{keyword}"].to_numpy(
            dtype=float
        )
        for keyword in keywords
    ]
)


valid = (
    np.isfinite(old_values)
    & np.isfinite(new_values)
    & (new_values > 0)
)


scale_factor = (
    np.sum(
        old_values[valid]
        * new_values[valid]
    )
    / np.sum(
        new_values[valid] ** 2
    )
)


print(
    "Common scale factor:",
    scale_factor,
)

Common scale factor: 1.0163614622863943


Bu yöntem ne yapıyor? Kabaca:

“Yeni veriyi hangi tek sayı ile çarparsam, 51 ortak haftada eski veriye mümkün olduğunca yaklaşırım?”

diye soruyor.

Sonra gerçekten fayda sağlıyor mu kontrol edeceğiz:

In [29]:
# --------------------------------------------------
# Scale uygulamadan önce / sonra fark
# --------------------------------------------------

for keyword in keywords:

    old = overlap[
        f"old_{keyword}"
    ].astype(float)

    new = overlap[
        f"new_{keyword}"
    ].astype(float)


    before = (
        old - new
    ).abs().mean()


    after = (
        old
        - new * scale_factor
    ).abs().mean()


    print(
        f"{keyword.upper()}: "
        f"before={before:.3f}, "
        f"after={after:.3f}"
    )

CHATGPT: before=1.922, after=1.584
GEMINI: before=0.706, after=0.762
CLAUDE: before=0.216, after=0.294


### Data Update Decision

Eski 3 yıllık haftalık veri ile yeni manuel Google Trends verisinin
51 ortak haftası karşılaştırıldı.

Ortak haftalardaki medyan ölçek oranı yaklaşık `1.0` olarak bulundu.
Ayrıca tek bir global scale factor hesaplandığında ChatGPT serisindeki
ortalama fark azalırken Gemini ve Claude serilerindeki hata arttı.

Bu nedenle yeni veri setine ek bir ölçek dönüşümü uygulanmamasına karar
verildi.

Mevcut 3 yıllık veri geçmiş referans olarak korunacak ve manuel olarak
indirilen haftalık veri setinden yalnızca eski veri setinde bulunmayan
yeni haftalar eklenecektir.

Bu yaklaşım, geçmiş analizlerde kullanılan veri setini değiştirmeden
güncel haftaların seriye eklenmesini sağlar.

In [30]:
# Şimdi yeni haftaları çıkar:

new_dates = new_data.index.difference(
    old_data.index
)

new_rows = new_data.loc[
    new_dates,
    [
        "chatgpt",
        "gemini",
        "claude",
    ],
].copy()


print("Yeni hafta sayısı:", len(new_rows))

display(new_rows)

Yeni hafta sayısı: 2


,chatgpt,gemini,claude
date,,,
2026-08-02,66,40,14
2026-08-09,70,40,15


Sonra birleştir:

In [31]:
updated_data = pd.concat(
    [
        old_data,
        new_rows,
    ]
).sort_index()


print("Eski son tarih:", old_data.index.max())
print("Yeni son tarih:", updated_data.index.max())

print("Eski satır sayısı:", len(old_data))
print("Yeni satır sayısı:", len(updated_data))

display(updated_data.tail(10))

Eski son tarih: 2026-07-26 00:00:00
Yeni son tarih: 2026-08-09 00:00:00
Eski satır sayısı: 157
Yeni satır sayısı: 159


,chatgpt,gemini,claude
date,,,
2026-06-07,71,47,21
2026-06-14,70,43,19
2026-06-21,68,43,18
2026-06-28,65,42,18
2026-07-05,65,39,17
2026-07-12,65,37,15
2026-07-19,69,38,16
2026-07-26,66,38,16
2026-08-02,66,40,14


In [32]:
updated_data.to_csv(
    "data/processed/google_trends_ai_3y_updated_2026-08-09.csv"
)

### Day 9 — Forecast Validation with New Actual Data

Dün, mevcut veri setinin son tarihi olan `2026-07-26` kullanılarak
ChatGPT, Gemini ve Claude için gelecek 4 haftalık tahminler üretilmişti.

Bugün Google Trends'ten manuel olarak indirilen güncel haftalık veride
`2026-08-02` ve `2026-08-09` haftalarının gerçek değerlerine artık sahibiz.

Bu nedenle önce yeni haftaları mevcut veri setine ekleyeceğiz. Ardından
dün üretilen tahminlerle bugün elde edilen gerçek değerleri karşılaştıracağız.

Bu karşılaştırmanın amacı, seçilen final modellerin gerçek gelecekte ilk iki
haftada ne kadar başarılı tahmin yaptığını görmek ve forecasting pipeline'ını
güncel veriyle doğrulamaktır.

In [33]:
# --------------------------------------------------
# Dün üretilen forecast'u yükle
# --------------------------------------------------

yesterday_forecast = pd.read_csv(
    "reports/final_forecast_as_of_2026-07-26.csv",
    parse_dates=["Date"],
)

yesterday_forecast = yesterday_forecast.set_index(
    "Date"
)

display(yesterday_forecast)

,ChatGPT,Gemini,Claude
Date,,,
2026-08-02,67.179036,37.824669,15.999439
2026-08-09,67.983816,37.848896,15.999473
2026-08-16,69.105038,37.845548,15.999471
2026-08-23,70.085163,37.846011,15.999471


In [34]:
# --------------------------------------------------
# Yeni gelen gerçek Google Trends değerleri
# --------------------------------------------------

actual_new = new_rows.rename(
    columns={
        "chatgpt": "ChatGPT",
        "gemini": "Gemini",
        "claude": "Claude",
    }
)

display(actual_new)

,ChatGPT,Gemini,Claude
date,,,
2026-08-02,66,40,14
2026-08-09,70,40,15


In [35]:
# --------------------------------------------------
# Tahmini yapılmış ve artık gerçeğini bildiğimiz haftalar
# --------------------------------------------------

validation_dates = (
    yesterday_forecast.index.intersection(
        actual_new.index
    )
)

print("Doğrulanabilen tarihler:")
print(validation_dates)

Doğrulanabilen tarihler:
DatetimeIndex(['2026-08-02', '2026-08-09'], dtype='datetime64[us]', freq=None)


In [36]:
# --------------------------------------------------
# Forecast vs Actual
# --------------------------------------------------

validation_rows = []

for keyword in [
    "ChatGPT",
    "Gemini",
    "Claude",
]:
    for date in validation_dates:

        actual = actual_new.loc[
            date,
            keyword,
        ]

        predicted = yesterday_forecast.loc[
            date,
            keyword,
        ]

        validation_rows.append(
            {
                "Date": date,
                "Trend": keyword,
                "Actual": actual,
                "Predicted": predicted,
                "Absolute_Error": abs(
                    actual - predicted
                ),
            }
        )


forecast_validation = pd.DataFrame(
    validation_rows
)

forecast_validation

,Date,Trend,Actual,Predicted,Absolute_Error
0,2026-08-02,ChatGPT,66,67.179036,1.179036
1,2026-08-09,ChatGPT,70,67.983816,2.016184
2,2026-08-02,Gemini,40,37.824669,2.175331
3,2026-08-09,Gemini,40,37.848896,2.151104
4,2026-08-02,Claude,14,15.999439,1.999439
5,2026-08-09,Claude,15,15.999473,0.999473


In [37]:
# --------------------------------------------------
# İlk gerçek forecast validation sonucu
# --------------------------------------------------

validation_summary = (
    forecast_validation
    .groupby("Trend")["Absolute_Error"]
    .mean()
    .sort_values()
    .to_frame("MAE")
)

validation_summary

,MAE
Trend,
Claude,1.499456
ChatGPT,1.597610
Gemini,2.163218


Buradaki MAE artık cross-validation MAE’sinden biraz farklı bir anlam taşıyor:

CV MAE: geçmişte yaptığımız simülasyonlardaki hata
Buradaki MAE: gerçekten geleceğe yaptığımız tahminin, veri geldikten sonra ölçülen hatası

Ama sadece 2 hafta olduğu için bundan “model kesin iyi/kötü” sonucu çıkarmayacağız. Şimdilik bunu ilk gerçek forecast doğrulaması olarak göreceğiz.

### First Real Future Forecast Validation

Day 8 sonunda `2026-07-26` tarihine kadar olan veriler kullanılarak
gelecek 4 haftalık tahminler üretilmişti.

Day 9'da Google Trends'ten alınan güncel haftalık veri sayesinde
`2026-08-02` ve `2026-08-09` haftalarının gerçek değerleri elde edildi
ve daha önce üretilen tahminlerle karşılaştırıldı.

İlk iki haftalık gerçek forecast performansı:

- Claude MAE: `1.499`
- ChatGPT MAE: `1.598`
- Gemini MAE: `2.163`

ChatGPT modelinin yükseliş yönünü doğru yakaladığı, Gemini modelinin
serinin sabit kalacağını doğru öngördüğü ancak seviyeyi bir miktar düşük
tahmin ettiği, Claude tahminlerinin ise gerçek değerlere yakın kaldığı
görüldü.

Bu değerlendirme yalnızca iki haftalık yeni veri içerdiği için kesin model
performansı sonucu olarak değil, ilk gerçek future forecast validation
deneyi olarak ele alınmıştır.

In [ ]:
forecast_validation.to_csv(
    "reports/forecast_validation_as_of_2026-08-09.csv",
    index=False,
)

### Re-evaluation with Updated Data

İki yeni haftalık gerçek veri (`2026-08-02` ve `2026-08-09`) veri setine
eklendikten sonra modeller aynı ayarlar değiştirilmeden yeniden
değerlendirilecektir.

Amaç yeniden hyperparameter tuning yapmak değil, yeni verinin daha önce
seçilen modelleri değiştirip değiştirmediğini kontrol etmektir.

Değerlendirme yine 12 fold × 4 haftalık time-series cross-validation ile
yapılacaktır.

In [38]:
import importlib
import src.forecasting

importlib.reload(src.forecasting)

from src.forecasting import (
    evaluate_naive_cv,
    evaluate_arima_cv,
    evaluate_prophet_cv,
    evaluate_xgb_recursive_with_change,
    evaluate_ensemble_cv,
    select_best_model,
)

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [39]:
# --------------------------------------------------
# Updated series
# --------------------------------------------------

N_LAGS = 8

chatgpt_updated = updated_data["chatgpt"]
gemini_updated = updated_data["gemini"]
claude_updated = updated_data["claude"]


print("Toplam hafta:", len(updated_data))
print("Son tarih:", updated_data.index.max())

Toplam hafta: 159
Son tarih: 2026-08-09 00:00:00


sadece feature hazırlama işini küçük bir yardımcı fonksiyona alalım

In [40]:
def prepare_xgb_data(
    series: pd.Series,
    n_lags: int = 8,
):
    """XGBoost için lag ve change feature'larını hazırlar."""

    lag_data = pd.DataFrame(
        {
            "target": series,

            **{
                f"lag_{i}": series.shift(i)
                for i in range(1, n_lags + 1)
            },
        }
    ).dropna()


    lag_data["change_1"] = (
        lag_data["lag_1"]
        - lag_data["lag_2"]
    )

    lag_data["change_2"] = (
        lag_data["lag_2"]
        - lag_data["lag_3"]
    )


    feature_columns = [
        *[
            f"lag_{i}"
            for i in range(1, n_lags + 1)
        ],
        "change_1",
        "change_2",
    ]


    X = lag_data[feature_columns]
    y = lag_data["target"]

    return X, y

Bu fonksiyon model kurmuyor. Sadece dün üç seri için ayrı ayrı yaptığımız:

lag_1 ... lag_8
change_1
change_2

hazırlığını tek yerde yapıyor.

In [41]:
X_chatgpt_updated, y_chatgpt_updated = prepare_xgb_data(
    chatgpt_updated
)

X_gemini_updated, y_gemini_updated = prepare_xgb_data(
    gemini_updated
)

X_claude_updated, y_claude_updated = prepare_xgb_data(
    claude_updated
)


print("ChatGPT:", X_chatgpt_updated.shape)
print("Gemini:", X_gemini_updated.shape)
print("Claude:", X_claude_updated.shape)

ChatGPT: (151, 10)
Gemini: (151, 10)
Claude: (151, 10)


In [43]:
from sklearn.model_selection import TimeSeriesSplit

In [44]:
tscv_updated = TimeSeriesSplit(
    n_splits=12,
    test_size=4,
)

#### ChatGPT

In [45]:
aligned_chatgpt_updated = chatgpt_updated.iloc[N_LAGS:]


naive_chatgpt_updated = evaluate_naive_cv(
    aligned_chatgpt_updated,
    tscv_updated,
)

arima_chatgpt_updated = evaluate_arima_cv(
    aligned_chatgpt_updated,
    tscv_updated,
    order=(1, 1, 1),
)

prophet_chatgpt_updated = evaluate_prophet_cv(
    aligned_chatgpt_updated,
    tscv_updated,
    changepoint_prior_scale=1.0,
    yearly_seasonality="auto",
)

xgb_chatgpt_updated = evaluate_xgb_recursive_with_change(
    X_chatgpt_updated,
    y_chatgpt_updated,
    tscv_updated,
)

ensemble_chatgpt_updated = evaluate_ensemble_cv(
    series=aligned_chatgpt_updated,
    X=X_chatgpt_updated,
    y=y_chatgpt_updated,
    splitter=tscv_updated,
    prophet_weight=0.5,
    xgb_weight=0.5,
    yearly_seasonality="auto",
)

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init

#### Gemini

In [46]:
aligned_gemini_updated = gemini_updated.iloc[N_LAGS:]


naive_gemini_updated = evaluate_naive_cv(
    aligned_gemini_updated,
    tscv_updated,
)

arima_gemini_updated = evaluate_arima_cv(
    aligned_gemini_updated,
    tscv_updated,
    order=(1, 1, 1),
)

prophet_gemini_updated = evaluate_prophet_cv(
    aligned_gemini_updated,
    tscv_updated,
    changepoint_prior_scale=1.0,
    yearly_seasonality="auto",
)

xgb_gemini_updated = evaluate_xgb_recursive_with_change(
    X_gemini_updated,
    y_gemini_updated,
    tscv_updated,
)

ensemble_gemini_updated = evaluate_ensemble_cv(
    series=aligned_gemini_updated,
    X=X_gemini_updated,
    y=y_gemini_updated,
    splitter=tscv_updated,
    prophet_weight=0.5,
    xgb_weight=0.5,
    yearly_seasonality="auto",
)

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameter

#### Claude

In [47]:
aligned_claude_updated = claude_updated.iloc[N_LAGS:]


naive_claude_updated = evaluate_naive_cv(
    aligned_claude_updated,
    tscv_updated,
)

arima_claude_updated = evaluate_arima_cv(
    aligned_claude_updated,
    tscv_updated,
    order=(1, 1, 1),
)

prophet_claude_updated = evaluate_prophet_cv(
    aligned_claude_updated,
    tscv_updated,
    changepoint_prior_scale=1.0,
    yearly_seasonality="auto",
)

xgb_claude_updated = evaluate_xgb_recursive_with_change(
    X_claude_updated,
    y_claude_updated,
    tscv_updated,
)

ensemble_claude_updated = evaluate_ensemble_cv(
    series=aligned_claude_updated,
    X=X_claude_updated,
    y=y_claude_updated,
    splitter=tscv_updated,
    prophet_weight=0.5,
    xgb_weight=0.5,
    yearly_seasonality="auto",
)

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init

#### otomatik seçim

In [48]:
updated_results = {}


updated_results["ChatGPT"] = select_best_model(
    {
        "Naive": naive_chatgpt_updated,
        "ARIMA": arima_chatgpt_updated,
        "Prophet": prophet_chatgpt_updated,
        "XGBoost": xgb_chatgpt_updated,
        "Ensemble": ensemble_chatgpt_updated,
    }
)


updated_results["Gemini"] = select_best_model(
    {
        "Naive": naive_gemini_updated,
        "ARIMA": arima_gemini_updated,
        "Prophet": prophet_gemini_updated,
        "XGBoost": xgb_gemini_updated,
        "Ensemble": ensemble_gemini_updated,
    }
)


updated_results["Claude"] = select_best_model(
    {
        "Naive": naive_claude_updated,
        "ARIMA": arima_claude_updated,
        "Prophet": prophet_claude_updated,
        "XGBoost": xgb_claude_updated,
        "Ensemble": ensemble_claude_updated,
    }
)

### Tablo:

In [50]:
updated_summary = pd.DataFrame(
    {
        trend: {
            "Best Model": result[0],
            "Mean MAE": result[1].iloc[0]["Mean_MAE"],
            "Mean RMSE": result[1].iloc[0]["Mean_RMSE"],
        }
        for trend, result in updated_results.items()
    }
).T


updated_summary

,Best Model,Mean MAE,Mean RMSE
ChatGPT,Ensemble,3.850029,4.610887
Gemini,Naive,4.333333,4.907285
Claude,Naive,1.583333,1.742705


Evet, burada **çok önemli bir şey oldu** ve sonuçların değişmesi aslında mantıklı. Özellikle Gemini’nin `ARIMA → Naive`, Claude’un da `ARIMA → Naive` dönmesi “iki yeni hafta bütün modeli bozdu” anlamına gelmiyor.

Sebebi bizim CV yapımız:

```python
TimeSeriesSplit(
    n_splits=12,
    test_size=4,
)
```

Her zaman **son 48 haftayı** test ediyor. Veri seti 149 satırken test edilen 48 haftalık pencere başka yerdeydi; şimdi 151 satıra çıkınca bütün fold’lar **2 hafta ileri kaydı**.

Gemini’de bu çok kritik. Önceden Fold 1:

```text
Train sonu: 2025-08-24 → Gemini = 13

Test:
2025-08-31 → 28
2025-09-07 → 63
2025-09-14 → 100
2025-09-21 → 60
```

Model spike hakkında **hiçbir şey bilmiyordu**. O yüzden bütün modeller çok büyük hata yapmıştı.

Şimdi iki yeni hafta eklenince ilk test dönemi yaklaşık olarak:

```text
Train artık şunları da görüyor:
2025-08-31 → 28
2025-09-07 → 63

Test:
2025-09-14 → 100
2025-09-21 → 60
2025-09-28 → 45
2025-10-05 → 37
```

oluyor.

Yani bizim o meşhur **Nano Banana şokunun ilk iki haftası artık training tarafına geçti.** Model artık “Gemini’de bir hareket başladı” bilgisini görüyor. Bu yüzden eski ve yeni CV sonuçlarını birebir aynı sınav gibi karşılaştırmamamız gerekiyor.

Claude’da da aynı şey var: test dönemlerinin tamamı iki hafta ileri kaydığı için Naive’ın avantajlı olduğu dönemlerin ağırlığı değişebiliyor.

Şimdiki sonuç:

```text
ChatGPT → Ensemble   MAE 3.850
Gemini  → Naive      MAE 4.333
Claude  → Naive      MAE 1.583
```

ama hemen:

> “Final modelleri artık Naive yapalım.”

demeyelim.

Önce **tüm model skorlarını görelim**, çünkü Naive ile ARIMA arasında belki `0.02` fark vardır. Winner sütunu bunu göstermiyor.



Sonra iki şeyi ayıracağız:

**Dünkü soru:**

> 26 Temmuz itibarıyla, o gün sahip olduğumuz bilgiyle hangi model en iyiydi?

Cevap:

```text
ChatGPT → Ensemble
Gemini  → ARIMA
Claude  → ARIMA
```

**Bugünkü soru:**

> 9 Ağustos’a kadar bilgi geldikten sonra, güncel son 48 haftalık değerlendirmede hangi model daha iyi?

Şu anda:

```text
ChatGPT → Ensemble
Gemini  → Naive
Claude  → Naive
```

Bu aslında projemiz açısından çok değerli bir sonuç: **model selection statik olmak zorunda değil; yeni veri geldikçe yeniden değerlendirilebilir.** Fakat model değişikliği yapmadan önce skor farklarının gerçekten anlamlı olup olmadığına bakacağız.


In [51]:


print("GEMINI")
display(
    updated_results["Gemini"][1]
)

print("\nCLAUDE")
display(
    updated_results["Claude"][1]
)

print("\nCHATGPT")
display(
    updated_results["ChatGPT"][1]
)



GEMINI


,Model,Mean_MAE,Mean_RMSE
0,Naive,4.333333,4.907285
1,Ensemble,6.890268,8.114654
2,XGBoost,8.278117,9.538647
3,Prophet,8.314925,9.144303
4,ARIMA,10.973544,12.944827



CLAUDE


,Model,Mean_MAE,Mean_RMSE
0,Naive,1.583333,1.742705
1,ARIMA,1.645413,1.800504
2,Ensemble,1.808664,1.946091
3,Prophet,1.819995,1.995245
4,XGBoost,2.004721,2.168254



CHATGPT


,Model,Mean_MAE,Mean_RMSE
0,Ensemble,3.850029,4.610887
1,XGBoost,3.855950,4.660631
2,Prophet,4.562816,5.263044
3,ARIMA,4.872044,5.676067
4,Naive,5.166667,6.018597


Evet, **ilk bakışta çok garip**, ama aslında ARIMA’nın çalışma biçimi yüzünden bu sonuç mümkün. Burada “iki yeni hafta geldi, ARIMA bir anda bozuldu”dan çok, **cross-validation pencerelerinin iki hafta kayması** kritik.

Dünkü Gemini CV’sinde ilk test fold’u kabaca şuydu:

```text
Train sonu:
... 12 → 13

Test:
28 → 63 → 100 → 60
```

ARIMA spike’ı hiç görmediği için yaklaşık 13 civarında kalıyordu ve o fold’da çok büyük hata yapıyordu.

Bugün iki hafta ekleyince bütün 12 fold iki hafta ileri kaydı. İlk fold artık yaklaşık şöyle:

```text
Train sonu:
13 → 28 → 63

Test:
100 → 60 → 45 → 37
```

İşte ARIMA açısından bu çok farklı bir durum.

ARIMA özellikle **son hareketlerden devam eden bir yapı çıkarmaya** çalışıyor. Son gördüğü değerler:

```text
13 → 28 → 63
```

gibi çok sert yükseliyorsa model:

> “Yükseliş devam ediyor olabilir.”

diye davranabiliyor.

Ama gerçekte:

```text
63 → 100 → 60 → 45 → 37
```

oluyor. Yani önce bir tepe, sonra çok hızlı düşüş var.

Dolayısıyla ARIMA burada sadece tepeyi kaçırmakla kalmayıp, **yükseliş momentumunu yanlış biçimde ileri taşıyıp düşüş tarafında da hata yapabilir**. Bu da tek bir fold’un hatasını çok büyütebilir.

Daha da önemlisi, önceki:

```text
ARIMA MAE ≈ 6.58
```

ile şimdiki:

```text
ARIMA MAE ≈ 10.97
```

**aynı 48 haftanın testi değil.** Son 48 haftalık test penceresi iki hafta ileri kaydı. Yani model aynı sınava tekrar girmedi; sınav soruları değişti.

Naive’ın neden şimdi `4.33` ile çok iyi olduğuna da bak: spike başladıktan sonra sadece:

> “Bir sonraki değer son gördüğüm değere yakın olsun.”

diyor.

Özellikle Gemini spike sonrasında uzun süre:

```text
45 → 37 → 34 → 32 → 31 → 31 → 32 → 35 ...
```

gibi nispeten kademeli hareket ettiği için Naive bu yeni test penceresinde çok avantajlı hale gelmiş olabilir.

Ama bunu tahmin ederek bırakmayalım. **ARIMA’nın hangi fold’da patladığını bulalım.** Şunu çalıştır:


**Ben büyük ihtimalle ilk bir-iki fold’da ARIMA’nın aşırı hata yaptığını düşünüyorum.** Onu görünce gerekirse o fold’un gerçek değerlerini ve ARIMA tahminlerini yan yana çıkarırız. Orada `6.58 → 10.97` sıçramasının tam nereden geldiğini çıplak gözle göreceğiz.


In [52]:

gemini_updated_fold_errors = pd.DataFrame(
    {
        "Fold": range(1, 13),
        "Naive": naive_gemini_updated["MAE"],
        "ARIMA": arima_gemini_updated["MAE"],
        "Prophet": prophet_gemini_updated["MAE"],
        "XGBoost": xgb_gemini_updated["MAE"],
        "Ensemble": ensemble_gemini_updated["MAE"],
    }
)

gemini_updated_fold_errors


,Fold,Naive,ARIMA,Prophet,XGBoost,Ensemble
0,1,21.00,103.565893,35.922028,20.443992,20.622680
1,2,5.00,2.128649,20.610809,29.744942,25.177876
2,3,2.50,2.512358,1.002382,22.199642,11.153477
3,4,1.75,1.750839,5.347624,1.803267,2.138109
4,5,4.75,4.677693,6.819957,3.030853,2.214033
5,6,1.50,1.645754,1.715684,1.691724,1.642368
6,7,4.00,2.460361,6.038075,6.378597,6.208336
7,8,1.75,2.921952,3.365190,1.162988,2.264089
8,9,3.25,3.442666,2.481493,3.913208,3.197350
9,10,1.75,2.021073,2.086287,1.320397,1.360281


In [53]:

gemini_updated_fold_errors.sort_values(
    by="ARIMA",
    ascending=False,
)


,Fold,Naive,ARIMA,Prophet,XGBoost,Ensemble
0,1,21.00,103.565893,35.922028,20.443992,20.622680
4,5,4.75,4.677693,6.819957,3.030853,2.214033
8,9,3.25,3.442666,2.481493,3.913208,3.197350
7,8,1.75,2.921952,3.365190,1.162988,2.264089
2,3,2.50,2.512358,1.002382,22.199642,11.153477
6,7,4.00,2.460361,6.038075,6.378597,6.208336
10,11,2.75,2.282612,6.231354,3.333729,4.782541
11,12,2.00,2.272679,8.158218,4.314064,1.922077
1,2,5.00,2.128649,20.610809,29.744942,25.177876
9,10,1.75,2.021073,2.086287,1.320397,1.360281


In [55]:
from statsmodels.tsa.arima.model import ARIMA

In [56]:
# --------------------------------------------------
# Gemini Updated - Fold 1
# Actual vs ARIMA prediction
# --------------------------------------------------

train_index, test_index = list(
    tscv_updated.split(aligned_gemini_updated)
)[0]


train = aligned_gemini_updated.iloc[
    train_index
]

test = aligned_gemini_updated.iloc[
    test_index
]


arima_fold1_model = ARIMA(
    train,
    order=(1, 1, 1),
)

arima_fold1_fitted = (
    arima_fold1_model.fit()
)


arima_fold1_prediction = (
    arima_fold1_fitted.forecast(
        steps=len(test)
    )
)


arima_fold1_comparison = pd.DataFrame(
    {
        "Actual": test.values,
        "ARIMA": arima_fold1_prediction.values,
    },
    index=test.index,
)


arima_fold1_comparison["Absolute_Error"] = (
    arima_fold1_comparison["Actual"]
    - arima_fold1_comparison["ARIMA"]
).abs()


arima_fold1_comparison

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameter

,Actual,ARIMA,Absolute_Error
date,,,
2025-09-14,100,104.151577,4.151577
2025-09-21,60,144.571452,84.571452
2025-09-28,45,184.272635,139.272635
2025-10-05,37,223.267906,186.267906


In [57]:
arima_fold1_comparison[
    "Absolute_Error"
].mean()

np.float64(103.5658925181274)

### Çıkarım:

Aynen, şimdi **103 MAE’nin nereden geldiği tamamen ortaya çıktı**. Senin tahminin doğruymuş: ARIMA gerçekten 100’ün çok üstüne çıkmış.

Fold 1’de olan şu:

```text
Tarih        Gerçek    ARIMA      Hata
09-14         100      104.15      4.15
09-21          60      144.57     84.57
09-28          45      184.27    139.27
10-05          37      223.27    186.27
```

İlk haftayı aslında **çok iyi yakalıyor**: `100` yerine `104`.

Ama sonra gerçek seri:

```text
100 → 60 → 45 → 37
```

diye çöküşe geçerken ARIMA:

```text
104 → 145 → 184 → 223
```

diye yükselişi devam ettiriyor. İşte MAE’nin patlamasının sebebi bu.

### ARIMA neden böyle yaptı?

Train verisinin sonunda model şunu görmüştü:

```text
13 → 28 → 63
```

Yani haftalık değişimler kabaca:

```text
+15
+35
```

gibi çok güçlü pozitif yönde.

`ARIMA(1,1,1)` içinde `d=1` kullandığımız için model doğrudan seviyelerden çok **haftadan haftaya değişimin davranışını** öğrenmeye çalışıyor.

Bu dönemde ona verilen son mesaj kabaca:

> “Gemini çok hızlı yükselmeye başladı.”

olduğu için ARIMA bu yükseliş hareketini ileri taşımış.

Modelin bilmediği şey şu:

> `100` Google Trends’in göreli maksimumu ve seri burada dönüp düşecek.

ARIMA’da doğal bir:

```text
0 ≤ tahmin ≤ 100
```

kısıtı yok. O yüzden `144`, `184`, `223` üretmesi matematiksel olarak tamamen mümkün.

Bu örnek ayrıca bize çok önemli bir şey gösteriyor: **ARIMA’nın güncel MAE’sinin 10.97 olmasının neredeyse tamamı tek bir anormal fold’dan geliyor.**

Fold 1’i dışarıda bırakırsak kalan 11 fold’daki ARIMA MAE ortalaması yaklaşık **2.56**. Yani:

```text
Tüm 12 fold → MAE ≈ 10.97
Fold 1 hariç → MAE ≈ 2.56
```

Bu bayağı büyük fark.

Dolayısıyla burada “ARIMA Gemini’de artık kötü bir model” demek çok kaba olur. Daha doğru yorum:

> **ARIMA normal dönemlerde Gemini serisini oldukça iyi tahmin edebiliyor; ancak ani ve güçlü bir yükselişin hemen ardından trendin tersine dönmesi durumunda yükselişi aşırı şekilde ileri taşıyarak çok büyük hata yapabiliyor.**

Naive’ın `4.33` ile birinci çıkmasının sebebi de burada anlaşılıyor. Naive o kadar “zeki” davranmadığı için:

```text
son değer neyse onu devam ettir
```

diyor ve ARIMA gibi `223`e kadar uçmuyor. Yani bazen **daha basit model, aşırı tepki vermediği için daha güvenli** oluyor.

Bence bu Fold 1’i notebook’ta kesin tutalım. Çünkü model karşılaştırmasının en öğretici örneklerinden biri oldu: **ortalama metriği tek bir ekstrem dönem ciddi biçimde bozabilir ve zaman serisi modelinin davranışını fold bazında incelemek gerekir.**


Şimdi hemen ARIMA’yı çöpe atmayacağız, Fold 1’i de silmeyeceğiz. Bu Fold 1 gerçek hayatta başımıza gelebilecek bir durum ve modelin zayıf noktasını gösteriyor.

Bence Day 9’da şu sırayla ilerleyelim:

Google Trends’in 0–100 sınırını modele uygulatmayı test edelim. Çünkü 223 gibi bir tahmin zaten fiziksel olarak anlamsız: Google Trends skoru 100’ü geçemez. Bu yüzden ARIMA tahminlerini sonradan 0–100 arasına kırpmak (clip) metodolojik olarak savunulabilir.
Aynı CV’yi bounded/clipped ARIMA ile yeniden çalıştıralım.
Sonra Gemini’de:
Naive
normal ARIMA
clipped ARIMA
Prophet
XGBoost
Ensemble
sonuçlarını karşılaştıralım.
Eğer Naive hâlâ belirgin biçimde daha iyiyse, güncel Gemini için Naive seçilir. “ARIMA dün iyiydi” diye zorlamayız.
Model seçiminin yeni veri geldikçe değişebileceğini pipeline’ın bir özelliği olarak kabul ederiz.

### Clipping Fold 1

In [58]:
arima_fold1_comparison["ARIMA_Clipped"] = (
    arima_fold1_comparison["ARIMA"]
    .clip(lower=0, upper=100)
)

arima_fold1_comparison["Clipped_Error"] = (
    arima_fold1_comparison["Actual"]
    - arima_fold1_comparison["ARIMA_Clipped"]
).abs()

display(arima_fold1_comparison)

print(
    "Original MAE:",
    arima_fold1_comparison["Absolute_Error"].mean()
)

print(
    "Clipped MAE:",
    arima_fold1_comparison["Clipped_Error"].mean()
)

,Actual,ARIMA,Absolute_Error,ARIMA_Clipped,Clipped_Error
date,,,,,
2025-09-14,100,104.151577,4.151577,100.0,0.0
2025-09-21,60,144.571452,84.571452,100.0,40.0
2025-09-28,45,184.272635,139.272635,100.0,55.0
2025-10-05,37,223.267906,186.267906,100.0,63.0


Original MAE: 103.5658925181274
Clipped MAE: 39.5


In [59]:
import importlib
import src.forecasting

importlib.reload(src.forecasting)

from src.forecasting import evaluate_arima_cv

In [60]:
# --------------------------------------------------
# Gemini Updated - Fold 1
# Actual vs ARIMA prediction
# --------------------------------------------------

train_index, test_index = list(
    tscv_updated.split(aligned_gemini_updated)
)[0]


train = aligned_gemini_updated.iloc[
    train_index
]

test = aligned_gemini_updated.iloc[
    test_index
]


arima_fold1_model = ARIMA(
    train,
    order=(1, 1, 1),
)

arima_fold1_fitted = (
    arima_fold1_model.fit()
)


arima_fold1_prediction = (
    arima_fold1_fitted.forecast(
        steps=len(test)
    )
)


arima_fold1_comparison = pd.DataFrame(
    {
        "Actual": test.values,
        "ARIMA": arima_fold1_prediction.values,
    },
    index=test.index,
)


arima_fold1_comparison["Absolute_Error"] = (
    arima_fold1_comparison["Actual"]
    - arima_fold1_comparison["ARIMA"]
).abs()


arima_fold1_comparison

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameter

,Actual,ARIMA,Absolute_Error
date,,,
2025-09-14,100,104.151577,4.151577
2025-09-21,60,144.571452,84.571452
2025-09-28,45,184.272635,139.272635
2025-10-05,37,223.267906,186.267906


In [61]:
print(
    "Normal ARIMA MAE:",
    arima_gemini_updated["MAE"].mean(),
)

print(
    "Clipped ARIMA MAE:",
    arima_gemini_clipped["MAE"].mean(),
)

print(
    "Clipped ARIMA RMSE:",
    arima_gemini_clipped["RMSE"].mean(),
)

Normal ARIMA MAE: 10.973544098478406


NameError: name 'arima_gemini_clipped' is not defined

In [62]:
from sklearn.model_selection import TimeSeriesSplit
from src.forecasting import evaluate_arima_cv


# Güncel Gemini serisi
N_LAGS = 8

gemini_updated = updated_data["gemini"]

aligned_gemini_updated = gemini_updated.iloc[N_LAGS:]


# Aynı CV yapısı
tscv_updated = TimeSeriesSplit(
    n_splits=12,
    test_size=4,
)


# Normal ARIMA
arima_gemini_updated = evaluate_arima_cv(
    series=aligned_gemini_updated,
    splitter=tscv_updated,
    order=(1, 1, 1),
)


# 0-100 arasında sınırlandırılmış ARIMA
arima_gemini_clipped = evaluate_arima_cv(
    series=aligned_gemini_updated,
    splitter=tscv_updated,
    order=(1, 1, 1),
    clip_range=(0, 100),
)


print(
    "Normal ARIMA MAE:",
    arima_gemini_updated["MAE"].mean(),
)

print(
    "Clipped ARIMA MAE:",
    arima_gemini_clipped["MAE"].mean(),
)

print(
    "Clipped ARIMA RMSE:",
    arima_gemini_clipped["RMSE"].mean(),
)

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameter

Normal ARIMA MAE: 10.973544098478406
Clipped ARIMA MAE: 5.634719721967788
Clipped ARIMA RMSE: 6.494488454625942


/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init

Bu sonuç çok açıklayıcı oldu:

```text
Normal ARIMA MAE   = 10.974
Clipped ARIMA MAE  =  5.635
Clipped ARIMA RMSE =  6.494
```

Yani `0–100` sınırı koymak ARIMA’nın hatasını **neredeyse yarıya indirdi**. Demek ki `144 → 184 → 223` gibi gerçek dünyada mümkün olmayan tahminler gerçekten skoru ciddi biçimde bozuyormuş.

Ama önemli olan şu: **clipping ARIMA’yı tamamen kurtarmadı.** Güncel Gemini’de Naive’ın MAE’si yaklaşık `4.333` idi. Dolayısıyla:

```text
Naive           4.333  ← hâlâ daha iyi
ARIMA Clipped   5.635
ARIMA Normal   10.974
```

Bu bize ARIMA’nın iki ayrı problemi olduğunu gösteriyor:

1. **100’ün üzerine taşması** → clipping bunu çözüyor.
2. **Trend dönüşünü görememesi** → clipping bunu çözemiyor.

Yani `13 → 28 → 63` yükselişini görünce ARIMA hâlâ devam eden yükseliş bekliyor; biz sadece “223 deme, en fazla 100 de” diyoruz. Gerçek seri ise `100 → 60 → 45 → 37` diye dönüyor.

Şimdi bence doğru metodolojik adım şu: Google Trends hedefimiz doğal olarak `0–100` aralığında olduğundan **tüm forecasting modellerine aynı 0–100 çıktı sınırını uygulayıp final karşılaştırmayı bir kez daha adil şekilde yapalım.** Naive zaten bu sınır içinde. ARIMA’yı yaptık. Prophet, XGBoost ve Ensemble tahminlerine de metric hesaplanmadan önce `clip(0, 100)` uygularız.

Eğer bundan sonra da Gemini’de Naive kazanırsa artık gönül rahatlığıyla:

> **Güncel Gemini verisi için en iyi model Naive.**

deriz. Ve bu, dün ARIMA seçmiş olmamızla çelişmez; yeni veri gelince CV test penceresi değişti ve model-selection sistemi gerçekten güncellendi. Bu aslında projenin güzel bir sonucu.


In [64]:
import src.forecasting

importlib.reload(src.forecasting)

<module 'src.forecasting' from '/Users/nihalapple/Desktop/trend-forecast-project/src/forecasting.py'>

In [65]:
from src.forecasting import (
    evaluate_arima_cv,
    evaluate_prophet_cv,
    evaluate_xgb_recursive_with_change,
    evaluate_ensemble_cv,
    select_best_model,
)

In [66]:
from sklearn.model_selection import TimeSeriesSplit

from src.forecasting import (
    evaluate_naive_cv,
    evaluate_arima_cv,
    evaluate_prophet_cv,
    evaluate_xgb_recursive_with_change,
    evaluate_ensemble_cv,
    select_best_model,
)


# --------------------------------------------------
# 1. Gemini updated series
# --------------------------------------------------

N_LAGS = 8

gemini_updated = updated_data["gemini"]

aligned_gemini_updated = (
    gemini_updated.iloc[N_LAGS:]
)


# --------------------------------------------------
# 2. XGBoost features
# --------------------------------------------------

lag_data = pd.DataFrame(
    {
        "target": gemini_updated,
        **{
            f"lag_{i}": gemini_updated.shift(i)
            for i in range(1, N_LAGS + 1)
        },
    }
).dropna()


lag_data["change_1"] = (
    lag_data["lag_1"]
    - lag_data["lag_2"]
)

lag_data["change_2"] = (
    lag_data["lag_2"]
    - lag_data["lag_3"]
)


feature_columns = [
    *[
        f"lag_{i}"
        for i in range(1, N_LAGS + 1)
    ],
    "change_1",
    "change_2",
]


X_gemini_updated = (
    lag_data[feature_columns]
)

y_gemini_updated = (
    lag_data["target"]
)


# --------------------------------------------------
# 3. Same TimeSeriesSplit
# --------------------------------------------------

tscv_updated = TimeSeriesSplit(
    n_splits=12,
    test_size=4,
)


# --------------------------------------------------
# 4. Naive
# --------------------------------------------------

naive_gemini_bounded = (
    evaluate_naive_cv(
        series=aligned_gemini_updated,
        splitter=tscv_updated,
    )
)


# --------------------------------------------------
# 5. ARIMA - clipped
# --------------------------------------------------

arima_gemini_bounded = (
    evaluate_arima_cv(
        series=aligned_gemini_updated,
        splitter=tscv_updated,
        order=(1, 1, 1),
        clip_range=(0, 100),
    )
)


# --------------------------------------------------
# 6. Prophet - clipped
# --------------------------------------------------

prophet_gemini_bounded = (
    evaluate_prophet_cv(
        series=aligned_gemini_updated,
        splitter=tscv_updated,
        changepoint_prior_scale=1.0,
        yearly_seasonality="auto",
        clip_range=(0, 100),
    )
)


# --------------------------------------------------
# 7. XGBoost - clipped
# --------------------------------------------------

xgb_gemini_bounded = (
    evaluate_xgb_recursive_with_change(
        X=X_gemini_updated,
        y=y_gemini_updated,
        splitter=tscv_updated,
        n_lags=N_LAGS,
        max_depth=2,
        n_estimators=300,
        learning_rate=0.03,
        clip_range=(0, 100),
    )
)


# --------------------------------------------------
# 8. Ensemble - clipped
# --------------------------------------------------

ensemble_gemini_bounded = (
    evaluate_ensemble_cv(
        series=aligned_gemini_updated,
        X=X_gemini_updated,
        y=y_gemini_updated,
        splitter=tscv_updated,
        n_lags=N_LAGS,
        prophet_weight=0.5,
        xgb_weight=0.5,
        changepoint_prior_scale=1.0,
        yearly_seasonality="auto",
        max_depth=2,
        n_estimators=300,
        learning_rate=0.03,
        clip_range=(0, 100),
    )
)


# --------------------------------------------------
# 9. Model comparison
# --------------------------------------------------

gemini_bounded_results = {
    "Naive": naive_gemini_bounded,
    "ARIMA Clipped": arima_gemini_bounded,
    "Prophet Clipped": prophet_gemini_bounded,
    "XGBoost Clipped": xgb_gemini_bounded,
    "Ensemble Clipped": ensemble_gemini_bounded,
}


best_gemini_bounded, gemini_bounded_comparison = (
    select_best_model(
        gemini_bounded_results
    )
)


print(
    "Best Gemini Model:",
    best_gemini_bounded,
)

display(
    gemini_bounded_comparison
)

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameter

Best Gemini Model: Naive


,Model,Mean_MAE,Mean_RMSE
0,Naive,4.333333,4.907285
1,ARIMA Clipped,5.634720,6.494488
2,Ensemble Clipped,6.890268,8.114654
3,XGBoost Clipped,8.278117,9.538647
4,Prophet Clipped,8.314925,9.144303


### Bounded Forecast Evaluation — Gemini

Google Trends scores are naturally bounded between 0 and 100. During
cross-validation, ARIMA produced unrealistic forecasts above 100 after
a sudden trend spike.

Therefore, a `[0, 100]` prediction constraint was applied consistently
to the forecasting models.

Updated Gemini results:

- Naive: MAE = 4.333
- ARIMA (clipped): MAE = 5.635
- Ensemble (clipped): MAE = 6.890
- XGBoost (clipped): MAE = 8.278
- Prophet (clipped): MAE = 8.315

Naive remained the best-performing model.

Clipping substantially improved ARIMA by preventing unrealistic
extrapolation, but it could not solve ARIMA's inability to anticipate
the reversal following the sharp Gemini trend spike.

Therefore, Naive is selected as the current best model for Gemini.

In [67]:
def evaluate_all_bounded_models(
    series,
    n_lags=8,
):
    """
    Bir Google Trends serisini tüm modellerle değerlendirir.

    Tüm tahminler 0-100 aralığında tutulur.
    """

    # ----------------------------------------------
    # Lag features
    # ----------------------------------------------

    lag_data = pd.DataFrame(
        {
            "target": series,
            **{
                f"lag_{i}": series.shift(i)
                for i in range(1, n_lags + 1)
            },
        }
    ).dropna()


    lag_data["change_1"] = (
        lag_data["lag_1"]
        - lag_data["lag_2"]
    )

    lag_data["change_2"] = (
        lag_data["lag_2"]
        - lag_data["lag_3"]
    )


    feature_columns = [
        *[
            f"lag_{i}"
            for i in range(1, n_lags + 1)
        ],
        "change_1",
        "change_2",
    ]


    X = lag_data[feature_columns]

    y = lag_data["target"]

    aligned_series = (
        series.iloc[n_lags:]
    )


    # ----------------------------------------------
    # Cross-validation
    # ----------------------------------------------

    splitter = TimeSeriesSplit(
        n_splits=12,
        test_size=4,
    )


    # ----------------------------------------------
    # Models
    # ----------------------------------------------

    naive = evaluate_naive_cv(
        series=aligned_series,
        splitter=splitter,
    )


    arima = evaluate_arima_cv(
        series=aligned_series,
        splitter=splitter,
        order=(1, 1, 1),
        clip_range=(0, 100),
    )


    prophet = evaluate_prophet_cv(
        series=aligned_series,
        splitter=splitter,
        changepoint_prior_scale=1.0,
        yearly_seasonality="auto",
        clip_range=(0, 100),
    )


    xgb = evaluate_xgb_recursive_with_change(
        X=X,
        y=y,
        splitter=splitter,
        n_lags=n_lags,
        max_depth=2,
        n_estimators=300,
        learning_rate=0.03,
        clip_range=(0, 100),
    )


    ensemble = evaluate_ensemble_cv(
        series=aligned_series,
        X=X,
        y=y,
        splitter=splitter,
        n_lags=n_lags,
        prophet_weight=0.5,
        xgb_weight=0.5,
        changepoint_prior_scale=1.0,
        yearly_seasonality="auto",
        max_depth=2,
        n_estimators=300,
        learning_rate=0.03,
        clip_range=(0, 100),
    )


    # ----------------------------------------------
    # Comparison
    # ----------------------------------------------

    model_results = {
        "Naive": naive,
        "ARIMA": arima,
        "Prophet": prophet,
        "XGBoost": xgb,
        "Ensemble": ensemble,
    }


    best_model, comparison = (
        select_best_model(
            model_results
        )
    )


    return best_model, comparison, model_results

In [68]:
best_chatgpt_updated, chatgpt_bounded_comparison, chatgpt_bounded_results = (
    evaluate_all_bounded_models(
        updated_data["chatgpt"]
    )
)

print(
    "Best ChatGPT Model:",
    best_chatgpt_updated,
)

display(
    chatgpt_bounded_comparison
)

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init

Best ChatGPT Model: Ensemble


,Model,Mean_MAE,Mean_RMSE
0,Ensemble,3.850029,4.610887
1,XGBoost,3.855950,4.660631
2,Prophet,4.562816,5.263044
3,ARIMA,4.872044,5.676067
4,Naive,5.166667,6.018597


Burada clipping sonucu değiştirmemiş. Bu da ChatGPT modellerinin CV sırasında zaten büyük ölçüde 0–100 aralığında tahmin yaptığını gösteriyor. ChatGPT için güncel en iyi model hâlâ Ensemble.

Şimdi aynı fonksiyonla Claude’u çalıştıralım:

In [69]:
best_claude_updated, claude_bounded_comparison, claude_bounded_results = (
    evaluate_all_bounded_models(
        updated_data["claude"]
    )
)

print(
    "Best Claude Model:",
    best_claude_updated,
)

display(
    claude_bounded_comparison
)

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init

Best Claude Model: Naive


,Model,Mean_MAE,Mean_RMSE
0,Naive,1.583333,1.742705
1,ARIMA,1.645413,1.800504
2,Ensemble,1.808664,1.946091
3,Prophet,1.819995,1.995245
4,XGBoost,2.004721,2.168254


Evet, doğru: Claude için de Naive en iyi model çıktı.

Şu an güncel ve bounded değerlendirmeden sonra üç seri için seçimimiz kesinleşti:

ChatGPT → Ensemble   MAE = 3.850
Gemini  → Naive      MAE = 4.333
Claude  → Naive      MAE = 1.583

Claude’da ARIMA da çok yakın:

Naive  → 1.583
ARIMA  → 1.645

Yani fark sadece yaklaşık 0.062. Ama bizim seçim kuralımız en düşük Mean MAE olduğu için Naive seçmek doğru.

Bu da ilginç bir sonuç aslında:

ChatGPT → daha karmaşık model işe yarıyor
Gemini  → basit model daha dayanıklı
Claude  → basit model yeterli

Her seriye aynı modeli zorla uygulamamamızın neden doğru olduğunu da göstermiş olduk.

Şimdi Day 9'da model-selection kısmı bitti ✅

Sıradaki işimiz güncellenmiş veriyle gerçek ileri tahmin üretmek.

In [70]:
import importlib
import src.forecasting

importlib.reload(src.forecasting)

from src.forecasting import (
    forecast_naive,
    forecast_prophet,
    forecast_xgb_recursive_with_change,
)

In [71]:
# --------------------------------------------------
# Gemini + Claude final forecasts
# --------------------------------------------------

FORECAST_STEPS = 4


gemini_final_forecast = forecast_naive(
    series=updated_data["gemini"],
    steps=FORECAST_STEPS,
)


claude_final_forecast = forecast_naive(
    series=updated_data["claude"],
    steps=FORECAST_STEPS,
)


print("Gemini Forecast")
display(gemini_final_forecast)

print("\nClaude Forecast")
display(claude_final_forecast)

Gemini Forecast


2026-08-16    40
2026-08-23    40
2026-08-30    40
2026-09-06    40
Freq: W-SUN, Name: naive_forecast, dtype: int64


Claude Forecast


2026-08-16    15
2026-08-23    15
2026-08-30    15
2026-09-06    15
Freq: W-SUN, Name: naive_forecast, dtype: int64

In [72]:
# --------------------------------------------------
# ChatGPT - Prophet forecast
# --------------------------------------------------

chatgpt_prophet_forecast = forecast_prophet(
    series=updated_data["chatgpt"],
    steps=FORECAST_STEPS,
    changepoint_prior_scale=1.0,
    yearly_seasonality="auto",
)


# Google Trends sınırı
chatgpt_prophet_values = (
    chatgpt_prophet_forecast["yhat"]
    .clip(lower=0, upper=100)
    .to_numpy()
)


# --------------------------------------------------
# ChatGPT - XGBoost forecast
# --------------------------------------------------

chatgpt_xgb_forecast = (
    forecast_xgb_recursive_with_change(
        series=updated_data["chatgpt"],
        steps=FORECAST_STEPS,
        n_lags=8,
    )
)

chatgpt_xgb_values = (
    chatgpt_xgb_forecast
    .clip(lower=0, upper=100)
    .to_numpy()
)

15:21:46 - cmdstanpy - INFO - Chain [1] start processing
15:21:46 - cmdstanpy - INFO - Chain [1] done processing


In [73]:
# --------------------------------------------------
# ChatGPT - Final Ensemble
# --------------------------------------------------

chatgpt_ensemble_values = (
    0.5 * chatgpt_prophet_values
    + 0.5 * chatgpt_xgb_values
)


chatgpt_ensemble_values = np.clip(
    chatgpt_ensemble_values,
    0,
    100,
)


chatgpt_final_forecast = pd.Series(
    chatgpt_ensemble_values,
    index=chatgpt_xgb_forecast.index,
    name="chatgpt",
)


display(chatgpt_final_forecast)

2026-08-16    70.248219
2026-08-23    70.850414
2026-08-30    71.373637
2026-09-06    72.022927
Freq: W-SUN, Name: chatgpt, dtype: float64

In [74]:
# --------------------------------------------------
# Final 4-week forecast
# --------------------------------------------------

final_forecast = pd.DataFrame(
    {
        "ChatGPT": chatgpt_final_forecast,
        "Gemini": gemini_final_forecast,
        "Claude": claude_final_forecast,
    }
)


final_forecast.index.name = "date"

display(final_forecast)

,ChatGPT,Gemini,Claude
date,,,
2026-08-16,70.248219,40,15
2026-08-23,70.850414,40,15
2026-08-30,71.373637,40,15
2026-09-06,72.022927,40,15


Buradaki yorumumuz şu: ChatGPT Ensemble modeli hafif ve düzenli bir yükseliş öngörüyor (70.25 → 72.02). Gemini ve Claude’da ise seçilen model Naive olduğu için son gözlenen değerin korunacağını söylüyor: Gemini 40, Claude 15. Bunlar gerçek arama sayıları değil, Google Trends’in göreli 0–100 skorları.

In [75]:
final_forecast.to_csv(
    "../reports/final_forecast_as_of_2026-08-09.csv"
)

print("Final forecast saved.")

OSError: Cannot save file into a non-existent directory: '../reports'

In [76]:
from pathlib import Path


# --------------------------------------------------
# Project root'u otomatik bul
# --------------------------------------------------

current_dir = Path.cwd()

if (current_dir / "src").exists():
    project_root = current_dir

elif (current_dir.parent / "src").exists():
    project_root = current_dir.parent

else:
    raise FileNotFoundError(
        "Project root bulunamadı."
    )


# --------------------------------------------------
# reports klasörünü garanti et
# --------------------------------------------------

reports_dir = project_root / "reports"

reports_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# Forecast'u kaydet
# --------------------------------------------------

output_path = (
    reports_dir
    / "final_forecast_as_of_2026-08-09.csv"
)

final_forecast.to_csv(
    output_path
)


print("Saved to:")
print(output_path)

Saved to:
/Users/nihalapple/Desktop/trend-forecast-project/reports/final_forecast_as_of_2026-08-09.csv


## Anomaly Spike Detection

In [77]:
# --------------------------------------------------
# Gemini anomaly detection
# --------------------------------------------------

gemini_series = updated_data["gemini"].copy()

WINDOW = 8


# O haftadan ÖNCEKİ 8 haftanın ortalaması
rolling_mean = (
    gemini_series
    .shift(1)
    .rolling(WINDOW)
    .mean()
)


# Önceki 8 haftadaki normal değişkenlik
rolling_std = (
    gemini_series
    .shift(1)
    .rolling(WINDOW)
    .std()
)


# Mevcut haftanın normal seviyeden ne kadar uzak olduğunu ölç
anomaly_score = (
    (gemini_series - rolling_mean)
    / rolling_std
)


anomaly_data = pd.DataFrame(
    {
        "Gemini": gemini_series,
        "Rolling_Mean": rolling_mean,
        "Anomaly_Score": anomaly_score,
    }
)


display(
    anomaly_data.loc[
        "2025-07-01":"2025-10-31"
    ]
)

,Gemini,Rolling_Mean,Anomaly_Score
date,,,
2025-07-06,11,10.750,0.282038
2025-07-13,10,11.000,-1.870829
2025-07-20,10,11.000,-1.870829
2025-07-27,11,10.875,0.195047
2025-08-03,12,10.875,1.755426
2025-08-10,12,10.875,1.755426
2025-08-17,12,11.000,1.322876
2025-08-24,13,11.125,2.246793
2025-08-31,28,11.375,15.674200


Burada shift(1) özellikle önemli: 2025-09-14 değerinin sıra dışı olup olmadığını hesaplarken 2025-09-14’ü kendi “normal geçmişinin” içine koymuyoruz. Sadece o tarihten önce bildiğimiz verilere bakıyoruz.

# Değerlerin Detaylı Analizi

Evet, burada artık sadece “kod çalıştı mı?” diye bakmıyoruz; **bir veri analisti gibi neden bu hesabı yaptığımızı ve çıkan sayının ne söylediğini** anlamak önemli.

Bizim burada kurduğumuz şey bir **anomaly detection / olağan dışı hareket tespiti** mekanizması. Tahmin modeli “gelecek hafta değer kaç olur?” diye sorarken, anomaly detection farklı bir soru soruyor:

> **“Bu hafta gördüğüm değer, bu serinin yakın geçmişteki normal davranışına göre olağan mı, yoksa sıra dışı mı?”**

Gemini’de bunu özellikle istememizin sebebi de gördüğümüz `13 → 28 → 63 → 100` sıçraması. ARIMA bu sıçramayı “devam eden trend” gibi algılayıp sonrasında aşırı tahmin yaptı. Anomaly detector ise daha erken bir aşamada “burada normal dışı bir şey oluyor” diye işaret verebilir.

---

## 1. Önce neyi “normal” kabul ediyoruz?

Kodda:

```python
WINDOW = 8
```

dedik.

Bu, bir haftayı değerlendirirken **ondan önceki 8 haftayı referans dönem** olarak kullanacağımız anlamına geliyor.

Mesela `2025-08-31` değerini değerlendirirken o haftayı kendi geçmişine dahil etmiyoruz. Ondan önceki yaklaşık 8 haftaya bakıyoruz.

Bu nedenle:

```python
gemini_series.shift(1)
```

kullandık.

`shift(1)` çok önemli.

Normalde seri:

```text
10
10
11
12
12
12
13
28
```

ise `shift(1)` yaptıktan sonra 28’in bulunduğu satırın geçmiş hesabında 28 yer almıyor.

Yani sistem:

> “28 normal mi?”

diye sorarken **28’i normalin tanımını oluşturmak için kullanmıyor.**

Bu, makine öğrenmesindeki **data leakage** mantığıyla aynı düşünce. Bir olay gerçekleşmeden önce elimizde olmayan bilgiyi kullanmıyoruz.

---

# 2. Neden 8 hafta?

Buradaki `8` sihirli veya matematiksel olarak “tek doğru” sayı değil.

Biz burada yaklaşık **2 aylık yakın geçmişi** temsil etsin diye seçtik.

Gemini gibi teknoloji trendlerinde çok eski davranış bazen bugünün normali için fazla alakasız olabilir.

Mesela 1 yılın ortalamasını alsaydık:

```text
2024...
2025...
```

gibi çok eski dönemler de bugünkü “normal seviyeyi” belirleyecekti.

Ama bizim istediğimiz şey:

> “Gemini son zamanlarda genellikle nasıl davranıyordu?”

Bu yüzden yakın bir pencere kullandık.

Burada önemli bir trade-off var:

| Window                  | Davranış                                                 |
| ----------------------- | -------------------------------------------------------- |
| Çok küçük, ör. 3 hafta  | Çok hassas olur, küçük hareketlere bile alarm verebilir  |
| Orta, ör. 8 hafta       | Yakın geçmişi temsil eder, bizim başlangıç tercihimiz    |
| Çok büyük, ör. 26 hafta | Daha stabil olur ama yeni davranışlara yavaş adapte olur |

Bir veri analisti `8`i seçip “tamam kesin budur” demez.

Sonradan örneğin:

```text
4 hafta
8 hafta
12 hafta
16 hafta
```

deneyip hangisinin gerçek önemli olayları daha iyi yakaladığını karşılaştırabiliriz.

Yani şu an `WINDOW=8` bir **başlangıç hiperparametresi**.

---

# 3. `Rolling_Mean` ne?

Kodumuz:

```python
rolling_mean = (
    gemini_series
    .shift(1)
    .rolling(WINDOW)
    .mean()
)
```

Bunun anlamı:

> Önceki 8 haftadaki Gemini skorlarının ortalaması kaç?

Mesela senin tablonda:

```text
2025-08-31

Gemini         = 28
Rolling_Mean   = 11.375
```

Bu çok anlamlı.

Sistem diyor ki:

> “Gemini son 8 haftada yaklaşık **11.4 civarında** dolaşıyordu. Ama bu hafta bir anda **28** oldu.”

Yani tek başına `28` sayısı çok şey anlatmıyor.

Ama:

```text
Normal ≈ 11
Şimdi  = 28
```

dediğimiz anda bunun ciddi bir hareket olduğunu anlayabiliyoruz.

Bu yüzden anomaly detection’da **ham değer yerine bağlam** çok önemli.

---

# 4. Peki sadece ortalamaya bakmak neden yetmez?

Çünkü serilerin doğal oynaklıkları farklı olabilir.

Diyelim iki seri var.

Birinci seri:

```text
10
10
11
10
10
11
```

İkincisi:

```text
5
17
8
20
4
15
```

İkisinin ortalaması benzer seviyelerde olabilir.

Ama ilk seri çok stabil, ikinci seri zaten sürekli oynuyor.

Birinci seride:

```text
10 → 20
```

çok sıra dışı olabilir.

İkinci seride `20` görmek ise normal davranışının bir parçası olabilir.

Bu yüzden sadece:

```text
Current - Mean
```

hesaplamıyoruz.

Aynı zamanda geçmişteki **standart sapmaya** da bakıyoruz.

Kodumuz:

```python
rolling_std = (
    gemini_series
    .shift(1)
    .rolling(WINDOW)
    .std()
)
```

Standart sapmayı burada basitçe:

> **“Son 8 haftada değerler ne kadar oynuyordu?”**

diye düşünebilirsin.

---

# 5. Anomaly Score nasıl hesaplanıyor?

Asıl merkez burası:

```python
anomaly_score = (
    (gemini_series - rolling_mean)
    / rolling_std
)
```

Matematiksel olarak bu bir tür **z-score**:

[
\text{Anomaly Score}
====================

\frac{\text{Current Value}-\text{Historical Mean}}
{\text{Historical Standard Deviation}}
]

Ama formülü ezberlemek yerine anlamını düşün.

Anomaly Score bize şunu söylüyor:

> **“Bu haftaki değer, yakın geçmişteki normal seviyeden kaç standart sapma uzakta?”**

---

## 6. Mesela 15.67 ne demek?

Senin tablonda:

```text
2025-08-31

Gemini          28
Rolling Mean    11.375
Anomaly Score   15.674
```

Bu inanılmaz yüksek bir skor.

Yani sistem kabaca:

> “28 değeri, yakın geçmişteki davranışa göre olağan seviyeden yaklaşık **15.7 standart sapma yukarıda**.”

diyor.

Normal bir seri davranışı açısından bu:

> 🚨 **çok güçlü anomali**

demek.

Dolayısıyla veri analisti bu satırı görünce:

> “Gemini tarafında 31 Ağustos civarında normal varyasyonla açıklanamayacak kadar büyük bir kırılma başlamış.”

der.

Burada analistin bir sonraki sorusu artık istatistiksel değil, **iş/domain sorusu** olur:

> “O tarihte ne oldu?”

Ürün lansmanı mı?

Yeni model mi?

Viral bir olay mı?

Haber mi?

Rakipte bir olay mı?

İşte anomaly detection ile dış olay analizi burada birleşiyor.

---

# 7. Sonraki hafta: `63`

Bak:

```text
2025-09-07

Gemini          63
Rolling Mean    13.500
Anomaly Score    8.316
```

İlginç bir şey var.

`63`, `28`den çok daha yüksek.

Ama anomaly score:

```text
28 → 15.67
63 → 8.32
```

**azalmış.**

İlk bakışta garip görünebilir.

Ama sebebi çok önemli.

31 Ağustos’taki `28` artık 7 Eylül için **geçmişin bir parçası** haline geldi.

Dolayısıyla yakın geçmiş:

```text
10
11
12
...
13
28
```

gibi daha oynak hale geldi.

Hem rolling mean yükseldi, hem de rolling standard deviation büyüdü.

Sistem artık:

> “Bu seri zaten hareketlenmeye başladı.”

diyor.

Bu yüzden `63` hâlâ aşırı anormal, ama ilk kırılma kadar “şok edici” değil.

---

# 8. `100` geldiğinde neden skor daha da düşüyor?

```text
2025-09-14

Gemini          100
Rolling Mean     20.125
Anomaly Score     4.373
```

`100` en yüksek değer olmasına rağmen anomaly score yalnızca `4.37`.

Çünkü geçmiş pencerenin içinde artık:

```text
28
63
```

gibi büyük değerler de bulunuyor.

Yani sistem yeni rejime adapte olmaya başlıyor.

Bu anomaly detection için beklediğimiz davranış.

Başlangıçta:

```text
11 → 28
```

çok büyük şok.

Sonra:

```text
28 → 63
```

hala büyük şok.

Sonra:

```text
63 → 100
```

artık sistem:

> “Tamam, burada güçlü bir yükseliş dönemi başladı.”

demeye başlıyor.

Bu aslında çok değerli.

Çünkü anomaly detector’ın amacı her yüksek değerde sonsuza kadar alarm çalmak değil.

**Yeni davranışı zamanla normal kabul etmeye başlaması gerekiyor.**

---

# 9. Çok kritik satır: `2025-08-24`

Bak:

```text
Gemini         13
Rolling Mean   11.125
Anomaly Score   2.247
```

Burada henüz büyük patlama başlamamış.

Ama skor `2.25`.

Bu bize şöyle bir sinyal verebilir:

> “Henüz kesin anomali değil ama normalden dikkat çekici biçimde yukarı çıkmaya başladı.”

Yani belki erken bir **warning** olabilir.

31 Ağustos:

```text
15.67
```

ise artık açık bir anomaly.

Böylece sistemi ileride örneğin:

```text
score < 2       → Normal
2 ≤ score < 3   → Warning
score ≥ 3       → Anomaly
```

şeklinde tasarlayabiliriz.

Ama **3 değerini henüz körü körüne seçmeyelim.**

İstatistikte `|z| > 3` sık kullanılan bir anomaly başlangıç kuralıdır, ama bizim serimiz klasik normal dağılımlı laboratuvar verisi değil.

Dolayısıyla veri analisti olarak threshold’u geçmiş olaylar üzerinde test etmemiz daha iyi olur.

---

# 10. Negatif anomaly score ne?

Mesela:

```text
2025-10-26

Gemini          31
Rolling Mean    49.875
Anomaly Score   -0.787
```

Negatif olması:

> Mevcut değerin yakın geçmiş ortalamasının altında olduğunu

söylüyor.

Yani:

```text
pozitif score → yukarı yönlü sıra dışılık
negatif score → aşağı yönlü sıra dışılık
```

Biz şu anda özellikle **trend spike**, yani ani yükseliş aradığımız için pozitif taraf bizim için daha önemli.

Ama örneğin:

```text
Anomaly Score = -5
```

görseydik bu da:

> 🚨 beklenmedik sert düşüş

olabilirdi.

---

# 11. Peki neden 21 Eylül'de `60` sadece 0.86?

Bu da çok öğretici.

```text
2025-09-21

Gemini          60
Rolling Mean    31.375
Anomaly Score    0.869
```

`60`, eski döneme göre hâlâ çok yüksek.

Ama son haftalarda sistem zaten:

```text
28
63
100
```

gördü.

Dolayısıyla standart sapma muazzam büyüdü.

Artık `60`:

> “Bu yeni yüksek oynaklık rejimi içinde çok şaşırtıcı değil.”

hale geldi.

Ve aslında istediğimiz de bu.

---

# Bir veri analisti bu tabloya nasıl bakar?

Bir veri analisti yalnızca:

> “En yüksek anomaly score hangisi?”

demez.

Şöyle bir hikâye çıkarır:

**Temmuz–Ağustos ortası:** Gemini yaklaşık `10–12` seviyelerinde oldukça stabil. Anomaly score genellikle düşük.

**24 Ağustos:** Değer `13`, anomaly score `2.25`. Hafif olağan dışı yükseliş sinyali ortaya çıkıyor.

**31 Ağustos:** `28` ve anomaly score `15.67`. Net bir structural shock / olağan dışı hareket başlangıcı.

**7 Eylül:** `63`, skor `8.32`. Şok devam ediyor ve trend güçlü şekilde hızlanıyor.

**14 Eylül:** `100`, skor `4.37`. Seri hâlâ olağan dışı yüksek fakat detector yeni yükseliş rejimine adapte olmaya başlıyor.

**21 Eylül sonrası:** Değerler düşmeye başlıyor. Rolling mean ise önceki spike nedeniyle hâlâ yüksek. Anomaly skorları normale yaklaşıyor.

Yani analist açısından olay:

```text
Normal regime
      ↓
Early warning
      ↓
Shock detected
      ↓
Continued shock
      ↓
New high-volatility regime
      ↓
Normalization / decline
```

şeklinde okunabilir.

---

# Burada kullandığımız “feature”lar hangileri?

Şu anda aslında üç temel feature ürettik:

```text
Rolling_Mean
Rolling_Std
Anomaly_Score
```

Ham Gemini değeri de var.

`Rolling_Mean` = yakın geçmişin beklenen seviyesi.

`Rolling_Std` = yakın geçmişin doğal oynaklığı.

`Anomaly_Score` = mevcut haftanın bunlara göre ne kadar sıra dışı olduğu.

Burada `WINDOW=8` ise feature değil; **parametre**.

Çünkü feature’ın nasıl hesaplandığını kontrol ediyor.

Aynı şekilde ileride belirleyeceğimiz:

```text
ANOMALY_THRESHOLD = 3
```

da parametre olacak.

---

# Bunun bizim tahmin sistemimize asıl katkısı ne?

Bence en önemli kısım bu.

Biz artık iki farklı sistemi yan yana kuruyoruz:

```text
FORECASTING
“Gelecek 4 hafta ne olacak?”
        +
ANOMALY DETECTION
“Şu anda normal dışı bir şey başladı mı?”
```

Örneğin normal durumda:

```text
Gemini forecast → 40
Anomaly → False
```

sistem sessiz çalışabilir.

Ama yeni haftada:

```text
Gerçek Gemini = 70
Beklenen normal ≈ 40
Anomaly Score = 6.2
```

gibi bir şey oluşursa:

```text
🚨 Significant trend spike detected
```

diyebiliriz.

Ve sistem burada:

> “Normal forecast şu anda daha az güvenilir olabilir çünkü seride ani bir rejim değişikliği algılandı.”

uyarısı bile verebilir.

İşte bu, yalnızca tahmin çıkaran basit bir notebook’tan daha güçlü bir **trend monitoring + forecasting engine** yaklaşımına götürüyor.

Ve Gemini’de ARIMA’nın yaşadığı problemi düşünürsek bunun neden faydalı olduğu çok net: model yükselişi mekanik olarak geleceğe uzatmıştı; anomaly detector ise aynı anda **“bu gördüğün şey normal trend değil, olağan dışı bir şok”** diye ayrı bir bilgi sağlıyor.


#### anomaly score’u otomatik alarma çevirme

In [78]:
ANOMALY_THRESHOLD = 3

anomaly_data["Is_Anomaly"] = (
    anomaly_data["Anomaly_Score"].abs()
    >= ANOMALY_THRESHOLD
)

display(
    anomaly_data.loc[
        anomaly_data["Is_Anomaly"]
    ]
)

,Gemini,Rolling_Mean,Anomaly_Score,Is_Anomaly
date,,,,
2023-12-03,3,1.000,inf,True
2024-02-11,3,1.250,3.780432,True
2024-02-18,4,1.375,3.528113,True
2024-05-12,5,4.000,inf,True
2024-08-25,5,4.000,inf,True
2024-11-17,6,5.000,inf,True
2025-05-18,10,8.000,3.741657,True
2025-05-25,11,8.375,3.528113,True
2025-08-31,28,11.375,15.674200,True


Tamam, şimdi neyi sorduğunu anladım. **`inf` sonradan oluşmadı.** Az önce baktığımız tabloyla şimdiki tablonun kapsadığı tarihler farklı.

İlkinde şunu göstermiştik:

```python
display(
    anomaly_data.loc[
        "2025-07-01":"2025-10-31"
    ]
)
```

Yani sadece **Temmuz–Ekim 2025** dönemine baktık. O dönemde `rolling_std` sıfır olmadığı için `inf` görmedik.

Sonra şunu yaptık:

```python
display(
    anomaly_data.loc[
        anomaly_data["Is_Anomaly"]
    ]
)
```

Bu ise artık **tüm 3 yıllık veri içindeki bütün anomaly satırlarını** gösteriyor.

Dolayısıyla daha önce ekranda hiç görmediğimiz:

```text
2023-12-03
2024-05-12
2024-08-25
2024-11-17
```

gibi eski tarihler de tabloya geldi. İşte o eski dönemlerde önceki 8 haftanın bazı değerleri tamamen aynı olduğu için `rolling_std = 0` olmuş ve:

```text
(Current - Mean) / 0 = inf
```

çıkmış.

Yani olay tam olarak:

```text
İlk tablo:
sadece 2025-07 → 2025-10
→ inf olan eski tarihler gösterilmedi

İkinci tablo:
tüm tarihlerde Is_Anomaly == True
→ eski inf satırları da görünür oldu
```

**`Is_Anomaly` kodu `inf` üretmedi; zaten hesaplanmış olan `inf` değerlerini filtre sonucunda önümüze çıkardı.**

Hatta doğrulamak istersen ilk tablomuzdaki aynı tarih aralığını tekrar filtreleyince yine `inf` görmeyeceksin:

```python
display(
    anomaly_data.loc[
        "2025-07-01":"2025-10-31"
    ]
)
```

Yani kodda gizemli bir değişiklik yok; sadece **görüntülediğimiz veri aralığını genişlettik.**
